In [1]:
from useful_functions import format_datetime64, get_daterange_str

import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime
from ICONProcessor import ICONDataGrid

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [2]:
def get_file(sfc=False, pl=False):
    if pl:
        suffix = 'pl'
        avg_str = ''
    elif sfc:
        suffix = 'ml__sfc'
        avg_str = '_30min'
    else:
        suffix = 'ml__lowest5'
        avg_str = '_30min'
    
      
    file = f'{experiment_path}/{exp}_LES{avg_str}_51m_{suffix}.nc'
    return file

## Script settings

In [3]:
scope = 'H2'

exp = 'v370_2030_v2'

In [4]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    experiment_id =f"hefex2/{exp}/exp_R3B15_51m"
elif scope == 'H3':
    scope_name = 'HEFEX III'
    experiment_id = f"hefex3/{exp}/exp_R3B15_51m"
else:
    print('Invalid scope!')

experiment_path = f"/work/bb1461/{experiment_id}/cdo/"

loc_file = data_dir + f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [5]:
df_locs = pd.read_csv(loc_file).sort_values('topo')
df_locs = df_locs[~df_locs['cell_id'].isna()]
df_locs

,New Name,Type,Lat,Lon,Comment,Alt,cell_id,topo
0,A255,AWS EC,46.810911,10.784094,MOMAA4. on debris,2556.0,34960.0,2654.089111
1,A260,AWS EC,46.809083,10.779727,MOMAA3,2605.0,37345.0,2690.791504
10,D262,AWS Met,46.808336,10.778370,TLOGGER,2623.0,37360.0,2722.822510
2,A267R,AWS EC,46.805415,10.776431,MOMAA9. Swiss line,2671.0,37442.0,2757.178711
3,A267L,AWS EC,46.806711,10.773102,MOMAA10. Swiss line,2672.0,37425.0,2762.057373
4,A267,AWS EC,46.805913,10.775135,MOMAA8. Swiss line,2671.0,37433.0,2762.127930
11,D269,AWS Wind,46.804296,10.773574,AWS,2692.0,37287.0,2792.991455
12,D273,AWS Met,46.801794,10.771454,TLOGGER,2725.0,35704.0,2798.971191
6,A275L,AWS EC,46.800647,10.768919,Grenoble,2741.0,35706.0,2802.588623
5,A275R,AWS EC,46.799891,10.771788,Grenoble,2749.0,36058.0,2808.865967


In [6]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat = np.array(df_locs['Lat'])
dim_loc_lon = df_locs['Lon']
dim_loc_cid = df_locs['cell_id'].astype(int)
dim_loc_ele = df_locs['topo']

## Load and prepare ICON data for locations

In [7]:
def create_data_array(ds, icon_var, height, name, time_dim='time'):
    # get time from dataset
    time = [ICONDataGrid.convert_float_dt(f) for f in ds.time.values]
    time_naive = [dt.replace(tzinfo=None) for dt in time]
    t_res = '30min averages (origin is the last value of the timeseries)'

    # get data_in for variable (all cell IDs)
    data = ds[icon_var].values[:, height, np.array(dim_loc_cid-1)]

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "cid": ("location", dim_loc_cid, {"long_name": "ICON Cell ID of location"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation on ICON grid", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": ds[icon_var].long_name,
            "standard_name": ds[icon_var].standard_name,
            "units": ds[icon_var].units,
            "temporal_resolution": t_res,
        }
    )
    return xr_data

In [8]:
# surface values
ds = xr.open_dataset(get_file(sfc=True))
xr_t_2m  = create_data_array(ds, 't_2m',  0, 't_2m')
xr_qv_2m = create_data_array(ds, 'qv_2m', 0, 'qv_2m')

# values from lowest model level 
ds = xr.open_dataset(get_file(sfc=False))
xr_t_5m  = create_data_array(ds, 'temp', -1, 't_5m')
xr_qv_5m = create_data_array(ds, 'qv',   -1, 'qv_5m')
xr_u_5m  = create_data_array(ds, 'u',    -1, 'u_5m')
xr_v_5m  = create_data_array(ds, 'v',    -1, 'v_5m')

# values from 500hPa pressure level
ds = xr.open_dataset(get_file(pl=True))
xr_u_500hPa  = create_data_array(ds, 'u', 0, 'u_500hPa')
xr_v_500hPa  = create_data_array(ds, 'v', 0, 'v_500hPa')

## Build dataset

In [9]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m' : xr_t_2m,
    't_5m' : xr_t_5m,
    'qv_2m': xr_qv_2m,                     
    'qv_5m': xr_qv_5m, 
    'u_5m' : xr_u_5m,
    'v_5m' : xr_v_5m,
    'u_500hPa': xr_u_500hPa,                     
    'v_500hPa': xr_v_500hPa, 
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="30min")
ds_out = ds_out.reindex(time=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset (51m resolution) for {scope_name} locations",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "experiment_id": experiment_id,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Experiment based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'icon/HEFEX{scope[1]}__ICON_{exp}_aws_locations__avg30min_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/icon/HEFEX2__ICON_v370_2030_v2_aws_locations__avg30min_20230815-20230826.nc.


In [10]:
ds_out

<xarray.Dataset> Size: 344kB
Dimensions:   (time: 557, location: 19)
Coordinates:
  * time      (time) datetime64[us] 4kB 2023-08-15T00:30:00 ... 2023-08-26T14...
  * location  (location) object 152B 'A255' 'A260' 'D262' ... 'A309' 'D316'
    lat       (location) float64 152B 46.81 46.81 46.81 ... 46.79 46.79 46.8
    lon       (location) float64 152B 10.78 10.78 10.78 ... 10.75 10.75 10.74
    cid       (location) int64 152B 34960 37345 37360 ... 35803 38570 71618
    ele       (location) float64 152B 2.654e+03 2.691e+03 ... 3.185e+03
Data variables:
    t_2m      (time, location) float32 42kB 279.5 279.3 279.2 ... nan nan nan
    t_5m      (time, location) float32 42kB 282.1 281.9 281.8 ... nan nan nan
    qv_2m     (time, location) float32 42kB 0.005952 0.005843 ... nan nan
    qv_5m     (time, location) float32 42kB 0.007152 0.007033 ... nan nan
    u_5m      (time, location) float32 42kB 2.906 2.309 2.3 ... nan nan nan
    v_5m      (time, location) float32 42kB 2.457 2.47 2.458 ... nan nan nan
    u_500hPa  (time, location) float32 42kB 0.9408 0.9142 0.9162 ... 15.55 15.62
    v_500hPa  (time, location) float32 42kB 3.09 3.12 3.109 ... 9.688 9.952
Attributes:
    title:          Sliced ICON-LES dataset (51m resolution) for HEFEX II loc...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-10-25
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    experiment_id:  hefex2/v370_2030_v2/exp_R3B15_51m
    StartTime:      2023-08-15 00:30:00
    StopTime:       2023-08-26 14:30:00
    comment:        Experiment based on glacier outlines from OGGM SSP370, ye...